# Dispersión del importe histórico y las ventas futuras — v09
Lourdes Lema · UDLA · Progreso 2

Complemento de la visualización v08. Solo entrenamiento inicial: 8.939 observaciones cliente–corte, 4.225 clientes distintos. No se entrenan modelos ni se utilizan validación o prueba.

En Colab, ejecutar en orden y cargar **Evidencias_cortes_v06.zip**. Al final se descarga **Evidencias_dispersion_v09.zip**. Esta copia pública no conserva salidas calculadas. La reproducción de este complemento en Colab queda pendiente y debe diferenciarse de los análisis exploratorios previamente documentados.

## Carga y controles
**Propósito:** comprobar que se utiliza la misma partición de v07 y v08.
**Operaciones:** leer solo el CSV de entrenamiento del ZIP y comprobar fechas, claves, conteos, valores finitos y fórmula del importe neto por compra.
**Resultado esperado:** 8.939 observaciones en los tres cortes de 2010.
**Interpretación:** si falla un control, detenerse y revisar la entrada. No hay imputación, eliminación o redondeo de datos.

In [ ]:
from pathlib import Path
import zipfile, io, json, hashlib, platform
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# Puede establecerse una ruta local; en Colab dejar None y cargar el ZIP.
RUTA_ENTRADA = None
if RUTA_ENTRADA is None:
    from google.colab import files
    cargados = files.upload()
    candidatos = [n for n in cargados if n.lower().endswith('.zip')]
    assert len(candidatos) == 1, 'Cargue únicamente Evidencias_cortes_v06.zip.'
    ruta = Path(candidatos[0])
else:
    ruta = Path(RUTA_ENTRADA)
with zipfile.ZipFile(ruta) as z:
    candidatos = [n for n in z.namelist() if Path(n).name == 'entrenamiento_inicial_v06.csv']
    assert len(candidatos) == 1, 'No se identificó un único CSV de entrenamiento.'
    contenido = z.read(candidatos[0])
df = pd.read_csv(io.BytesIO(contenido), dtype={'Customer ID': 'string'})
df['fecha_corte'] = pd.to_datetime(df['fecha_corte'], errors='raise')
predictores = ['recencia_dias', 'frecuencia_180d', 'diversidad_180d',
               'importe_neto_180d', 'importe_neto_por_compra_180d']
variables = predictores + ['referencia_neto_90d', 'objetivo_neto_90d']
assert len(df) == 8939 and set(df.columns) == set(variables + ['Customer ID','fecha_corte'])
assert not df.isna().any().any()
assert not df.duplicated(['Customer ID','fecha_corte']).any()
conteos = df.groupby('fecha_corte').size()
esperados = pd.Series([2652,2822,3465], index=pd.to_datetime(['2010-06-01','2010-09-01','2010-12-01']))
assert conteos.equals(esperados)
assert np.isfinite(df[variables].to_numpy(dtype=float)).all()
assert (df[variables].nunique() > 1).all()
assert (df['frecuencia_180d'] >= 1).all()
assert np.allclose(df['importe_neto_por_compra_180d'], df['importe_neto_180d']/df['frecuencia_180d'])
print('Controles aprobados. Observaciones:', len(df))
print('Clientes distintos:', df['Customer ID'].nunique())
print(conteos.to_string())

## Preparación de las tres vistas
**Propósito:** distinguir la relación entre magnitudes monetarias y la relación entre sus posiciones relativas.
**Operaciones:** seleccionar importe_neto_180d y objetivo_neto_90d; calcular Pearson y Spearman con rangos medios para empates; repetir los coeficientes por corte. Preparar rangos expresados como porcentaje de la posición dentro del conjunto (rango medio / n × 100).
**Resultado esperado:** Pearson agrupado cercano a 0,815 y Spearman cercano a 0,456, en concordancia con v07.
**Interpretación:** cada punto representa un cliente en un corte. Los rangos se usan únicamente como recurso descriptivo, sin alterar predictores ni establecer un nuevo ranking comercial. Una posición alta significa un importe relativamente elevado dentro de este conjunto; no es una probabilidad. Los saldos cero empatados comparten posición.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
salida=Path('salidas_dispersion_v09');salida.mkdir(exist_ok=True)
archivos=[]
xcol='importe_neto_180d'; ycol='objetivo_neto_90d'
x=df[xcol];y=df[ycol]
rx=x.rank(method='average',pct=True)*100
ry=y.rank(method='average',pct=True)*100
filas=[]
for corte,g in [('Agrupado',df)]+[(str(c.date()),g) for c,g in df.groupby('fecha_corte')]:
    a=g[xcol];b=g[ycol]
    filas.append({'corte':corte,'observaciones':len(g),'pearson':a.corr(b),
                  'spearman':a.rank(method='average').corr(b.rank(method='average'))})
coeficientes=pd.DataFrame(filas)
print(coeficientes.round(6).to_string(index=False))
p=salida/'coeficientes_dispersion_v09.csv';coeficientes.to_csv(p,index=False);archivos.append(p)
puntos=df[['Customer ID','fecha_corte',xcol,ycol]].copy()
puntos['posicion_historica_pct']=rx;puntos['posicion_futura_pct']=ry
assert len(puntos)==8939 and not puntos.isna().any().any()
p=salida/'puntos_dispersion_v09.csv';puntos.to_csv(p,index=False);archivos.append(p)

## Figura de dispersión
**Propósito:** explorar visualmente la asociación sin retirar extremos, negativos o ceros.
**Operaciones:** dibujar las mismas 8.939 observaciones en tres paneles, sin muestreo, desplazamiento aleatorio ni recta ajustada. Usar transparencia para facilitar la lectura; algunos puntos pueden superponerse exactamente.
**Resultado esperado:** una figura con tres paneles, exportada en PNG y PDF.
**Interpretación:**
- **A. Importes originales:** muestra toda la amplitud monetaria. Los valores grandes comprimen visualmente las observaciones próximas al origen.
- **B. Escala logarítmica simétrica:** conserva los signos y el cero. Ambos ejes son lineales de −100 a 100 GBP y logarítmicos fuera de ese intervalo. Las distancias no representan aumentos monetarios iguales. Cambia la representación, no los datos; el coeficiente de Pearson sigue siendo el calculado sobre los importes originales.
- **C. Posiciones relativas:** muestra si una posición histórica elevada coincide con una posición futura elevada. La línea horizontal producida por los ceros futuros responde a los empates. Una nube dispersa no permite asignar un resultado futuro individual con certeza.

La diferencia entre Pearson y Spearman no demuestra por sí sola que los extremos causen la correlación. Esta figura no es un análisis formal de influencia ni una evaluación fuera de muestra. Las observaciones repetidas de clientes y las ventanas superpuestas impiden tratarlas como muestras independientes para inferencia convencional.

In [ ]:
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
fig,axs=plt.subplots(1,3,figsize=(15,5.4))
color='#256d85'
for ax in axs:
    ax.grid(alpha=.15);ax.set_axisbelow(True)
for ax in axs[:2]:
    ax.scatter(x,y,s=10,alpha=.23,color=color,edgecolors='none',rasterized=True)
    ax.axhline(0,color='#777777',linewidth=.6);ax.axvline(0,color='#777777',linewidth=.6)
    ax.set_xlabel('Importe histórico neto · 180 días (GBP)')
    ax.set_ylabel('Ventas futuras netas · 90 días (GBP)')
axs[0].set_title('A. Importes originales')
axs[0].xaxis.set_major_formatter(FuncFormatter(lambda v,p:f'{v:,.0f}'.replace(',',' ')))
axs[0].yaxis.set_major_formatter(FuncFormatter(lambda v,p:f'{v:,.0f}'.replace(',',' ')))
axs[0].tick_params(axis='x',labelrotation=35)
axs[1].set_title('B. Escala logarítmica simétrica')
for eje in ['x','y']:
    getattr(axs[1],'set_'+eje+'scale')('symlog',linthresh=100)
    getattr(axs[1],eje+'axis').set_major_formatter(FuncFormatter(lambda v,p:f'{v:,.0f}'.replace(',',' ')))
axs[1].tick_params(axis='x',labelrotation=35)
axs[2].scatter(rx,ry,s=10,alpha=.23,color=color,edgecolors='none',rasterized=True)
axs[2].set(title='C. Posiciones relativas',xlabel='Posición histórica (%)',ylabel='Posición futura (%)',xlim=(0,100),ylim=(0,100))
r=coeficientes.iloc[0]
fig.suptitle(f'Importe histórico y ventas futuras · entrenamiento inicial\n8.939 observaciones · Pearson = {r.pearson:.3f} · Spearman = {r.spearman:.3f}',fontsize=13)
fig.text(.5,.015,'Mismas observaciones en los tres paneles. B: tramo lineal ±100 GBP. C: rangos medios para empates.',ha='center',fontsize=10)
fig.tight_layout(rect=[0,.07,1,.88])
for ext in ['png','pdf']:
    p=salida/('figura_dispersion_v09.'+ext);fig.savefig(p,dpi=200,bbox_inches='tight');archivos.append(p)
plt.show();plt.close(fig)

## Evidencias y límites
**Propósito:** conservar coeficientes, puntos representados, figuras y procedencia.
**Operaciones:** registrar huellas SHA-256, versiones y parámetros; comprimir las salidas de esta ejecución.
**Resultado esperado:** Evidencias_dispersion_v09.zip.
**Interpretación:** compartir el ZIP para verificar reproducción antes de incorporar la figura al informe. No se han cambiado los cinco predictores ni la referencia histórica de 90 días. No hay selección de modelos o hiperparámetros.

Fuentes técnicas ya consultadas (evitar duplicarlas en el informe): SciPy community. (s. f.). *pearsonr* y *spearmanr*. https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html y https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html. Matplotlib development team. (s. f.). *Symlog scale*. https://matplotlib.org/stable/gallery/scales/symlog_demo.html. Los coeficientes y gráficos son resultados propios.

In [ ]:
bitacora={
 'fecha_ejecucion_utc':datetime.now(timezone.utc).isoformat(),
 'sha256_zip':hashlib.sha256(ruta.read_bytes()).hexdigest(),
 'sha256_csv_entrenamiento':hashlib.sha256(contenido).hexdigest(),
 'observaciones':len(df),'clientes_distintos':int(df['Customer ID'].nunique()),
 'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'matplotlib':matplotlib.__version__,
 'variables':[xcol,ycol], 'vistas':['original','symlog base 10, linthresh 100 GBP en ambos ejes','rangos medios / n * 100'],
 'tratamiento':'Sin muestreo, jitter, filtros, imputación ni eliminación de extremos; sin modelos.',
 'v08':'Reproducción de las tablas de Colab verificada y gráficos PNG revisados.',
 'v09':'Comparación con reproducción externa pendiente.'}
p=salida/'bitacora_dispersion_v09.json';p.write_text(json.dumps(bitacora,ensure_ascii=False,indent=2));archivos.append(p)
zip_salida=Path('Evidencias_dispersion_v09.zip')
with zipfile.ZipFile(zip_salida,'w',zipfile.ZIP_DEFLATED) as z:
    for p in archivos:z.write(p,p.name)
print('Evidencias generadas:',zip_salida.name)
try:
    from google.colab import files
except ImportError:
    print('Ejecución local: ZIP disponible en',zip_salida.resolve())
else:
    files.download(str(zip_salida))